<a href="https://colab.research.google.com/github/DanyRE4020/Concentracion_IA_avanzada/blob/main/Notebook12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns, sklearn, torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
plt.rcParams['figure.figsize'] = [10, 4]
print('torch', torch.__version__, 'sklearn', sklearn.__version__)

torch 2.11.0+cpu sklearn 1.6.1


In [7]:
flights = sns.load_dataset('flights')
y = pd.Series(flights['passengers'].values,
              index=pd.date_range('1949-01-01', periods=144, freq='MS',), name='passengers')
assert len(y) == 144 and y.sum() == 40363
train, test = y[:'1958'], y['1959']
H = len(test)

def rmse(f,a): return float(np.sqrt(np.mean((np.asarray(f,float)-np.asarray(a,float))**2)))
def mape(f,a): return float(100 * np.mean(np.abs((np.asarray(f,float)-np.asarray(a,float))/np.asarray(a,float))))
results = {}

snaive = np.tile(train.values[-12:], 1)
results['Seasonal naive'] = (rmse(snaive,test),mape(snaive,test))

In [9]:
def make_windows(series, p):
    X, t = [], []
    for i in range(len(series) - p):
        X.append(series[i:i+p])
        t.append(series[i+p])
    return np.array(X), np.array(t)

def recursive_forecast(model, history, p, h):
    hist = list(np.asarray(history, float))
    out = []
    for _ in range(h):
        x = np.array(hist[-p:]).reshape(1, -1)
        nxt = float(model.predict(x)[0])
        out.append(nxt)
        hist.append(nxt)
    return np.array(out)

def z_of(Lser):
    Lser = np.asarray(Lser, float)
    return Lser[13:] - Lser[12:-1] - Lser[1:-12] + Lser[:-13]

def undo(z_future, L_hist):
    Lh = list(L_hist)
    for zt in z_future:
        Lh.append(zt + Lh[-1] + Lh[-12] - Lh[-13])
    return np.array(Lh[len(L_hist):])

p = 13
L = np.log(y.values)
Ltr = L[:len(train)]
ztr = z_of(Ltr)
mz, sz = ztr.mean(), ztr.std()
Xz, tz = make_windows((ztr - mz) / sz, p)

f_zmean = np.exp(undo(np.full(H, mz), Ltr))
results['Transform, no learning'] = (rmse(f_zmean, test), mape(f_zmean, test))